# Module 03: Cellular Automata & Particle Computation
Exploration of **Elementary Cellular Automata (256 Rules), Langton's Lambda, Hanson & Crutchfield (1997) Rule 54 Domain Filtering, Particle Velocities, and Collision Catalogs**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# ------------------------------------------------------------------------------
# 1. VECTORIZED 1D ELEMENTARY CELLULAR AUTOMATA ENGINE
# ------------------------------------------------------------------------------

def evolve_eca(rule_number, width=141, steps=120, init_type='random', seed=42):
    '''Evolves a 1D Elementary Cellular Automaton with periodic boundary conditions.'''
    rng = np.random.default_rng(seed)
    rule_bits = np.array([(rule_number >> i) & 1 for i in range(8)], dtype=np.uint8)
    grid = np.zeros((steps, width), dtype=np.uint8)

    if init_type == 'single_seed':
        grid[0, width // 2] = 1
    elif init_type == 'collision_setup' and rule_number == 54:
        # Construct clean period-4 ether background (1110...) with two colliding defects
        base_tile = np.array([1, 1, 1, 0], dtype=np.uint8)
        grid[0] = np.tile(base_tile, width // 4 + 1)[:width]
        # Inject left-moving and right-moving localized defects
        grid[0, width // 3 : width // 3 + 3] = [0, 1, 0]
        grid[0, 2 * width // 3 : 2 * width // 3 + 3] = [1, 0, 1]
    else:
        grid[0] = rng.integers(0, 2, size=width, dtype=np.uint8)

    for t in range(steps - 1):
        left = np.roll(grid[t], 1)
        center = grid[t]
        right = np.roll(grid[t], -1)
        neighborhood_idx = (left << 2) | (center << 1) | right
        grid[t + 1] = rule_bits[neighborhood_idx]

    langton_lambda = float(np.sum(rule_bits)) / 8.0
    return grid, langton_lambda

# ------------------------------------------------------------------------------
# 2. RULE 54 DOMAIN FILTER (HANSON & CRUTCHFIELD, 1997)
# ------------------------------------------------------------------------------

def apply_rule54_domain_filter(spacetime_grid):
    '''
    Applies a domain-specific regular-language transducer filter tailored to Rule 54.
    Rule 54's regular background ether consists of period-4 spatial tiles:
    Phase A: {1110, 1101, 1011, 0111} and Phase B: {0001, 0010, 0100, 1000}.
    Cells covered by valid ether tiles are masked to 0; localized defects/particles remain 1.
    '''
    valid_ether_words = {
        (1, 1, 1, 0), (1, 1, 0, 1), (1, 0, 1, 1), (0, 1, 1, 1),
        (0, 0, 0, 1), (0, 0, 1, 0), (0, 1, 0, 0), (1, 0, 0, 0)
    }
    steps, width = spacetime_grid.shape
    in_domain = np.zeros_like(spacetime_grid, dtype=bool)

    for t in range(steps):
        row = spacetime_grid[t]
        for i in range(width):
            word = (int(row[i]), int(row[(i + 1) % width]), int(row[(i + 2) % width]), int(row[(i + 3) % width]))
            if word in valid_ether_words:
                for offset in range(4):
                    in_domain[t, (i + offset) % width] = True

    particle_mask = (~in_domain).astype(np.uint8)
    return particle_mask

def extract_collision_catalog(particle_mask):
    '''Identifies active particle density, propagation tracks, and high-density collision vertices.'''
    steps, width = particle_mask.shape
    defect_fraction = float(np.mean(particle_mask))
    # Local 3x3 window density >= 5 indicates a multi-particle collision vertex
    collision_points = []
    for t in range(1, steps - 1):
        for i in range(1, width - 1):
            patch_sum = np.sum(particle_mask[t - 1 : t + 2, i - 1 : i + 2])
            if patch_sum >= 6 and particle_mask[t, i] == 1:
                collision_points.append((t, i))
    return defect_fraction, collision_points

# ------------------------------------------------------------------------------
# 3. INTERACTIVE PARTICLE COMPUTATION LABORATORY
# ------------------------------------------------------------------------------

def run_ca_particle_lab(rule_number=54, init_type='collision_setup', show_filter=True):
    grid, lam = evolve_eca(rule_number, width=128, steps=110, init_type=init_type, seed=42)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
    fig.patch.set_facecolor('#F8FAFC')

    # Panel 1: Raw Spacetime Diagram
    axes[0].imshow(grid, cmap='binary', interpolation='nearest', aspect='auto')
    axes[0].set_title(f'1. Raw Spacetime Diagram (Rule {rule_number} | lambda={lam:.3f})', fontsize=11.5, fontweight='bold')
    axes[0].set_xlabel('Lattice Site i')
    axes[0].set_ylabel('Time Step t (Top -> Bottom)')

    # Panel 2: Filtered Particle View or Damage Spreading Comparison
    if rule_number == 54 and show_filter:
        p_mask = apply_rule54_domain_filter(grid)
        defect_frac, collisions = extract_collision_catalog(p_mask)
        axes[1].imshow(p_mask, cmap='magma', interpolation='nearest', aspect='auto')
        if collisions:
            c_t, c_i = zip(*collisions)
            axes[1].scatter(c_i, c_t, s=18, facecolors='none', edgecolors='#38BDF8', linewidths=0.8, label='Collision Vertices')
            axes[1].legend(loc='upper right', facecolor='white')
        axes[1].set_title('2. Hanson-Crutchfield Domain Filter (Particles & Collisions)', fontsize=11.5, fontweight='bold')
        axes[1].set_xlabel('Lattice Site i')
        axes[1].set_ylabel('Time Step t')
        status_msg = f'Rule 54 Ether Filtered | Particle Density = {defect_frac:.3f} | Detected Collision Events = {len(collisions)}'
    else:
        # Compare single-bit perturbation (Hamming distance over time)
        grid_pert = grid.copy()
        grid_pert[0, grid.shape[1] // 2] ^= 1
        rule_bits = np.array([(rule_number >> i) & 1 for i in range(8)], dtype=np.uint8)
        for t in range(grid.shape[0] - 1):
            idx = (np.roll(grid_pert[t], 1) << 2) | (grid_pert[t] << 1) | np.roll(grid_pert[t], -1)
            grid_pert[t + 1] = rule_bits[idx]
        diff = np.bitwise_xor(grid, grid_pert)
        axes[1].imshow(diff, cmap='Reds', interpolation='nearest', aspect='auto')
        axes[1].set_title('2. Perturbation / Damage Spreading (XOR Difference)', fontsize=11.5, fontweight='bold')
        axes[1].set_xlabel('Lattice Site i')
        axes[1].set_ylabel('Time Step t')
        status_msg = f'Rule {rule_number} | Note: Rule 54 Domain Filter is language-specific; showing perturbation spread for Rule {rule_number}.'

    plt.tight_layout()
    plt.show()
    print('=' * 85)
    print(status_msg)
    print('Known Rule 54 Particle Velocities: alpha (v = 0), beta (v = 0), gamma+ (v = +1/2), gamma- (v = -1/2)')
    print('=' * 85)

widgets.interact(
    run_ca_particle_lab,
    rule_number=widgets.Dropdown(options=[('Rule 54 (Particle Logic)', 54), ('Rule 110 (Turing Universal)', 110), ('Rule 30 (Chaotic)', 30), ('Rule 90 (Fractal/Linear)', 90)], value=54, description='ECA Rule:'),
    init_type=widgets.Dropdown(options=['collision_setup', 'random', 'single_seed'], value='collision_setup', description='Initial State:'),
    show_filter=widgets.Checkbox(value=True, description='Apply Rule 54 Domain Filter'),
);
